Build a nano GPT Part-02
-----
[video](https://youtu.be/kCc8FmEb1nY?t=2533)

<ins>Goal</ins>

Questions:
1. Why is that we are averaging past context for self attention

##Self-Attention Intro


Self-attention is a mechanism used in neural networks, particularly in transformers, that allows the model to weigh the importance of different tokens (which can be words or characters) in the input sequence. Instead of processing tokens in isolation, self-attention enables the model to consider the context provided by all other tokens in the sequence simultaneously. This is really useful for things like understanding language and even predicting the next character in a text.

**Key Points of Self-Attention:**

*   **Contextual Understanding:** It helps the model understand the meaning of a token based on its relationship with other tokens in the same sequence.
*   **Parallel Processing:** Unlike traditional sequential models (like RNNs), self-attention can process all tokens in a sequence in parallel, making it more efficient for long sequences.
*   **Variable Dependencies:** It can capture long-range dependencies between tokens, regardless of their distance in the sequence.
*   **Learned Weights:** The importance given to each token is learned during the training process.
*   **Core of Transformers:** Self-attention is a fundamental component of the Transformer architecture, which has achieved state-of-the-art results in various NLP tasks, including character-level language modeling.

**What is Bag-of-Words?**

Bag-of-words is a much simpler way to represent text. It just counts how many times each word appears in a document. It completely ignores the order of the words and the relationships between them.

For example, the sentences "The quick brown fox jumps over the lazy dog" and "The lazy dog jumps over the quick brown fox" would have the same bag-of-words representation because they use the same words the same number of times, even though their meaning is different.

**Is Bag-of-Words used in self-attention?**

No, bag-of-words is **not** used in self-attention. Self-attention, as explained above, specifically *does* consider the relationships and order of tokens in the sequence through the Query, Key, and Value mechanism. This is a key difference that makes self-attention (and models like transformers that use it) much more powerful for understanding context and generating coherent text compared to models based on bag-of-words.


**Simplified working of self-attention**

let's use the sentence "The dog ate the bone because it was hungry." to explain the point-by-point breakdown of self-attention. In this sentence, "it" refers to "dog".

1.  **Looking at Each Token:** The computer looks at each token: "The", "dog", "ate", "the", "bone", "because", "it", "was", "hungry", "."

2.  **Asking Questions (Queries), Getting Information (Keys), and Having the Content (Values):** For *each* of these ten tokens, the computer creates a Query, a Key, and a Value. Think of them like this for the token "dog":
    *   **Query ("dog"):** "What information about other tokens is relevant to understanding 'dog'?"
    *   **Key ("dog"):** "Here is the information I can offer about 'dog'."
    *   **Value ("dog"):** The actual meaning or representation of the token "dog".

    Let's look at the token "it" as another example:
    *   **Query ("it"):** "What information about other tokens is relevant to understanding 'it'?"
    *   **Key ("it"):** "Here is the information I can offer about 'it'."
    *   **Value ("it"):** The actual meaning or representation of the token "it".

3.  **Comparing and Scoring:** This is where the magic happens. The computer takes the Query of *one* token (let's say, the Query for "it") and compares it to the Key of *every other token* in the sentence.
    *   Query ("it") vs. Key ("The") -> Score 1
    *   Query ("it") vs. Key ("dog") -> Score 2 (This score will be high!)
    *   Query ("it") vs. Key ("ate") -> Score 3
    *   Query ("it") vs. Key ("the") -> Score 4
    *   Query ("it") vs. Key ("bone") -> Score 5
    *   Query ("it") vs. Key ("because") -> Score 6
    *   Query ("it") vs. Key ("it") -> Score 7 (This score will also be high!)
    *   Query ("it") vs. Key ("was") -> Score 8
    *   Query ("it") vs. Key ("hungry") -> Score 9
    *   Query ("it") vs. Key (".") -> Score 10

    This comparison is done for the Query of *every* token against the Key of *every* other token.

4.  **Weighting the Importance:** The scores from the comparison step are converted into weights. For "it", the weight associated with "dog" will be significantly higher than the weights for most other words because the comparison score was high.

5.  **Combining Information:** To create the new, context-aware representation for "it", the computer takes the Value of *every* token in the sentence and adds them up, but each Value is multiplied by the attention weight calculated in the previous step. Since the weight for "dog" was high when considering "it", the Value of "dog" will contribute a lot to the final representation of "it". This is how the model learns that "it" in this sentence refers to "dog".

This same process of comparing Queries and Keys, getting scores, converting to weights, and combining weighted Values is done for every single token in the input sequence simultaneously. This allows the model to build a rich understanding of how all the tokens relate to each other within their context.

###**Optional: The mathematical trick in self-attention**

The trick here is to find the weighted average of previous characters and there by showing us a very basic mechanism for how a model could incorporate information from the past(how much influence each past character has).

Note: The weighted average of previous characters is a simplified way to understand a core idea behind self-attention, but self-attention itself is more sophisticated.

In [58]:
import torch
import torch.nn as nn
from torch.nn import functional as F
# consider the following toy example:
torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [44]:
# option 1: finding weighted average/mean of past chars - using for loop
xbow = torch.zeros(B,T,C)
for b in range(B):
  for t in range(T):
    xprev = x[b,:t+1]
    xbow[b,t] = torch.mean(xprev, 0)
print(x[0])
print(xbow[0])

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])
tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


In [56]:
# option 2: finding weighted average of past chars - using matrix multiplication
wei = torch.tril(torch.ones(T,T)) # creates a lower triangular matrix of ones
sum = wei.sum(1, keepdim=True) # calculates the sum of each row
wei = wei / wei.sum(1, keepdim=True) # normalizes the wei matrix by dividing each element by its row sum

# calculates the weighted average of past characters for each position in the sequence across all batches
xbow2 = wei @ x # here wei.shape is [T,T] and x.shape is [B,T,C] , so during matrix multiplication wei will be broadcasted to [B,T,T]
torch.allclose(xbow, xbow2) # ideally this should be true, but might show false due to differences in floating-point precision

False

In [59]:
# option 3: finding weighted average of past chars - using softmax
tril = torch.tril(torch.ones(T, T))# creates a lower triangular matrix of ones
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf')) # masked_fill tells set inf to all elements in tril where its value is 0
wei = F.softmax(wei, dim=-1) # softmax exponentiate each element and divide by sum
xbow3 = wei @ x
torch.allclose(xbow, xbow3)# ideally this should be true, but might show false due to differences in floating-point precision

False